In [50]:
print("OK")

OK


In [51]:
%pwd

'c:\\Users'

In [52]:
import os 
os.chdir("../")

In [53]:
%pwd

'c:\\'

In [54]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [55]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [56]:
extracted_data = load_pdf_files("data")

In [57]:
extracted_data

[]

In [58]:
len(extracted_data)

0

In [59]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [60]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [61]:
minimal_docs

[]

In [62]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [63]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 0


In [64]:
texts_chunk

[]

In [65]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [66]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [67]:
vector = embedding.embed_query("Hello world")
vector

[-0.034477267414331436,
 0.031023245304822922,
 0.006735002622008324,
 0.02610897272825241,
 -0.03936199098825455,
 -0.16030247509479523,
 0.06692401319742203,
 -0.006441466510295868,
 -0.047450482845306396,
 0.014758829958736897,
 0.07087530195713043,
 0.05552758648991585,
 0.01919335499405861,
 -0.026251377537846565,
 -0.010109559632837772,
 -0.02694047801196575,
 0.022307416424155235,
 -0.02222663350403309,
 -0.1496925950050354,
 -0.01749303564429283,
 0.0076762172393500805,
 0.054352302104234695,
 0.003254439914599061,
 0.0317259281873703,
 -0.0846213549375534,
 -0.0294059868901968,
 0.051595572382211685,
 0.048124056309461594,
 -0.003314792178571224,
 -0.058279186487197876,
 0.04196929186582565,
 0.022210659459233284,
 0.128188818693161,
 -0.022338882088661194,
 -0.011656327173113823,
 0.06292837858200073,
 -0.03287631645798683,
 -0.0912260040640831,
 -0.031175334006547928,
 0.052699558436870575,
 0.04703483358025551,
 -0.08420311659574509,
 -0.0300561785697937,
 -0.02074487134814

In [68]:
print( "Vector length:", len(vector))

Vector length: 384


In [69]:
from dotenv import load_dotenv
import os
load_dotenv()

False

In [78]:
import os
from dotenv import load_dotenv

print("Current directory:", os.getcwd())
print(".env exists:", os.path.exists(".env"))
print("load_dotenv():", load_dotenv())

print(os.getenv("PINECONE_API_KEY"))

Current directory: c:\
.env exists: False
load_dotenv(): False
None


In [79]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print(PINECONE_API_KEY)
print(OPENAI_API_KEY)

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

None
None


TypeError: str expected, not NoneType

In [ ]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data to the existing Pinecone index

In [ ]:
dswith = Document(
    page_content="Add more data from any new source related to this",
    metadata={"source": "any"}
)

In [ ]:
docsearch.add_documents(documents=[dswith])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what dswithbappy?"})
print(response["answer"])